# Exercice 3 — Lire un taux et un score de rachat : incertitude, Brier et calibration

##  Contexte métier
L'équipe ALM vous demande de **lire de façon critique** deux choses : (1) les taux de rachat par segment présentés dans un reporting — sont-ils fiables ? (2) deux modèles de score de rachat proposés par la data science — lequel utiliser pour projeter des flux ? Règle de la formation : *« un chiffre sans effectif ni intervalle n'est pas un résultat, c'est une anecdote »*.

## Objectifs
- Calculer un taux de rachat par segment avec son **intervalle de confiance** (Wald puis Wilson)
- Vérifier par simulation que l'incertitude décroît en **1/√n** (théorème central limite)
- Montrer qu'une **AUC identique** peut cacher une **calibration** très différente (Brier)
- Construire un tableau de calibration et mesurer l'impact sur un **seuil métier** (p > 0,4)

##  Rappels théoriques mobilisés (slides de la session)
- *Échantillonnage et incertitude : la fiabilité d'un taux* — IC, segments rares
- *Probabilités : les lois qui décrivent l'actuariat vie* — espérance, variance, théorème central limite
- *Métriques* — score de Brier vs AUC ; *Calibration : quand une probabilité dit vrai*
- *Quatre erreurs de lecture* — « l'AUC est de 0,9, la proba est fiable » ; « 5 % sur le segment, c'est le taux »

##  Mini-dictionnaire & prérequis
| Élément | Signification / usage |
|---|---|
| `p̂ = k / n` | taux observé : `k` rachats parmi `n` contrats |
| IC de Wald à 95 % | p̂ ± 1,96 · √(p̂(1−p̂)/n) |
| IC de Wilson à 95 % | centre = (p̂ + z²/2n)/(1 + z²/n) ; demi-largeur = z·√(p̂(1−p̂)/n + z²/4n²)/(1 + z²/n), avec z = 1,96 |
| Brier | BS = moyenne de (p − y)² ; plus petit = meilleur ; mesure ranking **et** niveau |
| AUC (`roc_auc_score`) | capacité de classement uniquement, aucune information sur le niveau des probabilités |
| Calibration | parmi les contrats notés 0,30, ~30 % doivent racheter |
| `pd.qcut`, `groupby().agg()` | découper en classes de même effectif ; agréger par classe |

##  Mode d'emploi
- Complétez les cellules marquées `...  # À COMPLÉTER` puis exécutez-les **dans l'ordre**.
- Les cellules **Vérification** vous disent immédiatement si vous êtes sur la bonne piste (✅ / ❌ + indice).
- Bloqué ? Ouvrez les **💡 indices** progressifs avant de demander la correction.
- À la fin, la cellule `bilan()` donne votre score.

In [ ]:
# ✅ Outil d'auto-vérification — NE PAS MODIFIER
import numpy as np, pandas as pd, matplotlib.pyplot as plt, hashlib
_resultats = {}
def verifier(nom, condition, indice=""):
    """Évalue `condition` (une fonction sans argument) ; affiche ✅ ou ❌ + indice."""
    try: ok = bool(condition())
    except Exception: ok = False
    _resultats[nom] = ok
    print(("✅ " if ok else "❌ ") + nom + ("" if ok or not indice else "\n    💡 " + indice))
def bilan():
    n, k = len(_resultats), sum(_resultats.values())
    print(f"\n🏁 Score : {k}/{n} vérifications réussies")
    print("   Bravo, exercice terminé !" if k == n and n > 0 else "   Reprenez les ❌ ci-dessus (relancez la cellule concernée puis `bilan()`).")
def _h(s): return hashlib.sha1(s.encode()).hexdigest()[:8]
%matplotlib inline

---
## Partie A — Un taux de rachat sans intervalle est une anecdote

In [ ]:
from sklearn.metrics import roc_auc_score
segments = pd.DataFrame({
    "segment": ["Petits encours (< 10 k€)", "Encours moyens", "Gros encours (> 150 k€)"],
    "effectif": [4000, 1200, 40],
    "rachats": [200, 66, 2]})
segments

**Q1.** (a) Ajoutez la colonne `taux` = rachats / effectif. (b) Calculez l'intervalle de **Wald** à 95 % (`ic_bas`, `ic_haut`). (c) Que constatez-vous sur la dernière ligne ?

<details><summary>💡 Indice 1</summary>

`segments['taux'] = segments['rachats'] / segments['effectif']`

</details>

<details><summary>💡 Indice 2</summary>

`marge = 1.96 * np.sqrt(taux * (1 - taux) / effectif)` ; `ic_bas = taux - marge`.

</details>

In [ ]:
segments["taux"] = ...        # À COMPLÉTER
marge = ...                   # À COMPLÉTER : demi-largeur de Wald (Series)
segments["ic_bas"] = ...      # À COMPLÉTER
segments["ic_haut"] = ...     # À COMPLÉTER
segments.round(4)

In [ ]:
verifier("Q1a · taux = rachats/effectif", lambda: np.allclose(segments["taux"], [0.05, 0.055, 0.05]))
verifier("Q1b · IC de Wald, segment 1", lambda: np.isclose(segments["ic_bas"][0], 0.043246, atol=1e-4) and np.isclose(segments["ic_haut"][0], 0.056754, atol=1e-4))
verifier("Q1b · IC de Wald, gros encours", lambda: np.isclose(segments["ic_bas"][2], -0.017542, atol=1e-4))

**Q1d.** On utilise donc l'intervalle de **Wilson** (formule dans le dictionnaire). Écrivez `ic_wilson(k, n, z=1.96)` qui renvoie `(bas, haut)`, puis calculez `bas_gros, haut_gros = ic_wilson(2, 40)`.

In [ ]:
def ic_wilson(k, n, z=1.96):
    ...   # À COMPLÉTER : renvoie (bas, haut)

bas_gros, haut_gros = ic_wilson(2, 40)
print(f"Gros encours : {2/40:.1%} [{bas_gros:.1%} ; {haut_gros:.1%}]")

In [ ]:
verifier("Q1d · Wilson borne basse > 0", lambda: bas_gros > 0)
verifier("Q1d · Wilson ≈ [1,4 % ; 16,5 %]", lambda: np.isclose(bas_gros, 0.01382, atol=1e-3) and np.isclose(haut_gros, 0.165042, atol=1e-3))

## Partie B — Pourquoi la taille d'échantillon protège (1/√n)

**Q2.** Pour p = 5 %, simulez 2 000 échantillons de taille n ∈ {40, 400, 4 000} (`rng.binomial(n, p, size=2000) / n`) et calculez l'écart-type empirique des taux estimés → dictionnaire `ecarts_types`. Comparez à la théorie √(p(1−p)/n) → `theorie`.

<details><summary>💡 Indice 1</summary>

`rng = np.random.default_rng(1)` ; pour chaque `n` : `taux_simules = rng.binomial(n, p, size=2000) / n` puis `.std()`.

</details>

In [ ]:
rng = np.random.default_rng(1)
p = 0.05
ecarts_types, theorie = {}, {}
for n in [40, 400, 4000]:
    ...   # À COMPLÉTER : remplir ecarts_types[n] et theorie[n]
print(pd.DataFrame({"empirique": ecarts_types, "théorique": theorie}).round(4))

In [ ]:
verifier("Q2 · 3 tailles simulées", lambda: set(ecarts_types) == {40, 400, 4000})
verifier("Q2 · empirique ≈ théorique (±10 %)", lambda: all(abs(ecarts_types[n]/theorie[n] - 1) < 0.10 for n in (40, 400, 4000)))
verifier("Q2 · ×100 d'effectif ⇒ incertitude ÷10", lambda: 8 < ecarts_types[40] / ecarts_types[4000] < 12, "√100 = 10")

## Partie C — Deux scores de rachat : même AUC, mêmes probabilités ?

La data science propose deux modèles pour une **population à risque** (5 000 contrats). Le modèle A est calibré par construction ; le modèle B a exactement le même classement, mais des probabilités gonflées.

In [ ]:
rng = np.random.default_rng(7)
N = 5000
p_vrai = rng.beta(2, 9, N)                 # probabilité « réelle » de rachat
y = rng.binomial(1, p_vrai)                # rachat observé (0/1)
p_A = p_vrai                               # modèle A
p_B = p_vrai ** 0.55                       # modèle B : même ordre, niveaux gonflés
print(f"Taux de rachat observé : {y.mean():.1%} | proba moyenne A : {p_A.mean():.1%} | proba moyenne B : {p_B.mean():.1%}")

**Q3.** Calculez `auc_A`, `auc_B` (avec `roc_auc_score`) et `brier_A`, `brier_B` (à la main : `np.mean((p - y)**2)`). Puis répondez : pour **projeter des flux de rachat**, quel modèle choisir ? (`choix_modele` = `"A"` ou `"B"`).

<details><summary>💡 Indice 1</summary>

Le Brier est la moyenne des carrés des écarts entre probabilité prédite et issue 0/1.

</details>

In [ ]:
auc_A = ...     # À COMPLÉTER
auc_B = ...     # À COMPLÉTER
brier_A = ...   # À COMPLÉTER
brier_B = ...   # À COMPLÉTER
choix_modele = ...   # "A" ou "B"
print(f"AUC  A={auc_A:.4f}  B={auc_B:.4f}\nBrier A={brier_A:.4f}  B={brier_B:.4f}")

In [ ]:
verifier("Q3 · AUC identiques (même classement)", lambda: np.isclose(auc_A, auc_B, atol=1e-9), "B est une transformation croissante de A")
verifier("Q3 · Brier de A", lambda: np.isclose(brier_A, 0.134935, atol=1e-5))
verifier("Q3 · Brier de B plus mauvais", lambda: brier_B > brier_A and np.isclose(brier_B, 0.173476, atol=1e-5))
verifier("Q3 · choix pour projeter des flux", lambda: choix_modele == "A", "il faut des probabilités calibrées")

**Q4 · Tableau de calibration.** Écrivez `table_calibration(p, y, q=5)` : découpez les contrats en `q` classes d'effectif égal selon `p` (`pd.qcut(p, q, labels=False)`) et renvoyez un DataFrame indexé par classe avec `p_moy` (probabilité moyenne), `taux_obs` (taux observé) et `effectif`. Appliquez-la aux deux modèles.

<details><summary>💡 Indice 1</summary>

`df = pd.DataFrame({'p': p, 'y': y}); df['classe'] = pd.qcut(df['p'], q, labels=False)`

</details>

<details><summary>💡 Indice 2</summary>

`df.groupby('classe').agg(p_moy=('p','mean'), taux_obs=('y','mean'), effectif=('y','size'))`

</details>

In [ ]:
def table_calibration(p, y, q=5):
    ...   # À COMPLÉTER

tab_A, tab_B = table_calibration(p_A, y), table_calibration(p_B, y)
ecart_A = (tab_A.p_moy - tab_A.taux_obs).abs().max()
ecart_B = (tab_B.p_moy - tab_B.taux_obs).abs().max()
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.plot([0, .6], [0, .6], "k--", label="calibration parfaite")
ax.plot(tab_A.p_moy, tab_A.taux_obs, "o-", label="Modèle A"); ax.plot(tab_B.p_moy, tab_B.taux_obs, "s-", label="Modèle B")
ax.set_xlabel("probabilité prédite moyenne"); ax.set_ylabel("taux de rachat observé"); ax.legend(); plt.show()
print(f"Écart max à la diagonale : A={ecart_A:.3f}  B={ecart_B:.3f}"); tab_B.round(3)

In [ ]:
verifier("Q4 · 5 classes d'effectif égal", lambda: tab_A.shape == (5, 3) and tab_A.effectif.nunique() == 1)
verifier("Q4 · colonnes attendues", lambda: list(tab_A.columns) == ["p_moy", "taux_obs", "effectif"])
verifier("Q4 · A proche de la diagonale, B non", lambda: ecart_A < 0.04 and ecart_B > 2 * ecart_A, "comparez p_moy et taux_obs")

**Q5 · Impact sur un seuil métier.** Une action commerciale est déclenchée si `p > 0,4`. Pour chaque modèle, calculez le nombre de contrats ciblés (`nb_A`, `nb_B`) et le **taux de rachat réellement observé** parmi eux (`taux_reel_A`, `taux_reel_B`).

In [ ]:
seuil = 0.4
cible_A, cible_B = p_A > seuil, p_B > seuil
nb_A, nb_B = ..., ...                         # À COMPLÉTER : nombre de contrats ciblés
taux_reel_A, taux_reel_B = ..., ...           # À COMPLÉTER : y.mean() parmi les ciblés
print(f"A : {nb_A} ciblés, taux réel {taux_reel_A:.1%} | B : {nb_B} ciblés, taux réel {taux_reel_B:.1%}")

In [ ]:
verifier("Q5 · nombres ciblés", lambda: nb_A == 244 and nb_B == 2057)
verifier("Q5 · B cible beaucoup plus de contrats", lambda: nb_B > 2 * nb_A)
verifier("Q5 · le seuil « 0,4 » n'a pas le même sens", lambda: taux_reel_A > 0.38 and taux_reel_B < 0.38 - 0.05, "comparez au seuil de 40 %")

In [ ]:
bilan()